In [1]:
import sys

import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as wgt
import rich
import tqdm.auto as tqdm

from IPython.display import display
from pathlib import Path

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib widget

In [3]:
src_directory = Path('../src')
init_directory = src_directory / 'initialization'
src_path = Path('../src')
sys.path.append(str(src_path))

print = rich.print # nicer outputs

In [4]:
from plotting.splinetools import SplineSettingsDashboard
from plotting.curveditor import MonotonicCurveEditor
from plotting.simulator import SimulationController, SimulationControllerDashboard, SequenceParameters
from plotting.scattercanvas import InteractiveRelaxometricParameterCanvas
from plotting.signaldisplay import SignalDisplayDashboard, SignalDisplay, wire_callbacks
from plotting.historyplot import HistoryPlot

from slsqp import optimize_sequence

import seqmetrics as seqmetrics


In [5]:
FA_DATA_PATH = init_directory / 'fa_cao.npy'
TR_DATA_PATH = init_directory / 'tr_cao.npy'
fa = np.load(FA_DATA_PATH)
tr = np.load(TR_DATA_PATH)
fa_initial_y = fa
fa_initial_x = np.arange(len(fa))
tr_initial_y = tr
tr_initial_x = np.arange(len(tr))

In [6]:
dashboard_fa = SplineSettingsDashboard.from_FA_defaults()
dashboard_tr = SplineSettingsDashboard.from_TR_defaults()

seqparams = SequenceParameters(
    ph=np.full_like(fa, fill_value=0.0),
    shots=len(fa),
    prep=[1],
    t2te=[0.0],
    ti=[10.0],
    te=1.0
)

with plt.ioff():
    fig, axes = plt.subplots(ncols=3, figsize=(12, 4))
    ce_fa = MonotonicCurveEditor(fa_initial_x, fa_initial_y, dashboard_fa, fig=fig, ax=axes[0])
    ce_fa.ax.set_ylabel('Flip Angle (degrees)')
    ce_fa.ax.set_title('Interactive Flip Angle Editor')

    ce_tr = MonotonicCurveEditor(tr_initial_x, tr_initial_y, dashboard_tr, fig=fig, ax=axes[1], initial_yaxis_range=(0, 100))
    ce_tr.ax.set_ylabel('Repetition Time (ms)')
    ce_tr.ax.set_title('Interactive Repetition Time Editor')

    cv = InteractiveRelaxometricParameterCanvas.prepopulated(species={'csf', 'wm', 'gm', 'muscle'}, ax=axes[2], fig=fig)

    ctr_dashboard = SimulationControllerDashboard()

    controller = SimulationController(
        dashboard=ctr_dashboard,
        fa_provider=ce_fa,
        tr_provider=ce_tr,
        relax_provider=cv,
        parameters=seqparams
    )

    tabs = wgt.Tab(
        children=[ctr_dashboard.ui, dashboard_fa.ui, dashboard_tr.ui],
        titles=['Simulation Dashboard', 'FA Settings', 'TR Settings'],
        style={'description_width': 'initial'}
    )

    _ = fig.tight_layout()
    
controller.run()

In [7]:
dashboard = SignalDisplayDashboard.create()
sigdisp = SignalDisplay()
sigdisp.fig.update_layout(height=400, width=1400, showlegend=False)
sigdisp.fig.update_layout(xaxis=dict(title='Time (ms)'), yaxis=dict(title='Signal Amplitude (a.u.)'))

sigdisp.add_traces(controller.fetch_simulation_package(), dashboard.query_state())
wire_callbacks(sigdisp, dashboard, controller)

In [8]:
display(
    wgt.VBox([
        wgt.VBox([tabs, fig.canvas]),
        dashboard.ui,
        sigdisp.fig
    ])
)

With the setup obtained above, we can optimize the sequence

In [10]:
T1 = np.asarray([p.x for p in cv.get_points()], dtype=np.float32)
T2 = np.asarray([p.y for p in cv.get_points()], dtype=np.float32)
M0 = 1.0
beats = seqparams.beats
shots = seqparams.shots
prep = seqparams.prep
ti = [10] #seqparams.ti
t2te = [1] #seqparams.t2te
te = 10.0
ph = seqparams.ph
fa_min = 5.0
fa_max = 90.0
fa_maxdiff = 5.0
fa = ce_fa.get_current_curve().y
tr = ce_tr.get_current_curve().y

In [11]:
init_fa = ce_fa.get_current_curve().y

hp = HistoryPlot(max_TTL=40)
hp.add_immortal_trace(init_fa, width=1, color='green', dash='dot', opacity=0.7)
hp.fig.update_layout(xaxis=dict(title='TR index'), yaxis=dict(title='Flip Angle (degrees)'))

FigureWidget({
    'data': [{'line': {'color': 'green', 'dash': 'dot', 'width': 1},
              'meta': {'TTL': -999, 'is_immortal': True},
              'mode': 'lines',
              'name': 'immortal',
              'opacity': 0.7,
              'type': 'scatter',
              'uid': '4e8fbdc0-5bbd-423b-965c-8f34fd96f9ea',
              'y': {'bdata': ('MDHd+rOiJEDRWXRTEEMlQMibzlhi4y' ... '/7QyRAm3HHIuxUJEB7S2vi+2UkQA=='),
                    'dtype': 'f8'}}],
    'layout': {'height': 400,
               'showlegend': True,
               'template': '...',
               'width': 1200,
               'xaxis': {'title': {'text': 'TR index'}},
               'yaxis': {'title': {'text': 'Flip Angle (degrees)'}}}
})

In [12]:
n_max_iter: int = 40

pbar = tqdm.tqdm(total=n_max_iter)

def callback(x: np.ndarray):
    # tgriesler optimizatiopn: single array for fa and tr
    # only propagate flip angles into visualization
    hp.add_trace(x[:x.size//2])
    pbar.update(1)

result = optimize_sequence(
    costfunction='orth_epg',
    t1=T1,
    t2=T2,
    m0=M0,
    beats=beats,
    shots=shots,
    fa=init_fa,
    tr=tr,
    ph=ph,
    prep=prep,
    ti=ti,
    t2te=t2te,
    te=te,
    fa_min=fa_min,
    fa_max=fa_max,
    fa_maxdiff=fa_maxdiff,
    n_iter_max=n_max_iter,
    callback=callback,
    iprint=0
)

  0%|          | 0/40 [00:00<?, ?it/s]

In [ ]:
DATA = [t.y for t in hp.fig.data]


  0%|          | 0/200 [00:00<?, ?it/s]

  NIT    FC           OBJFUN            GNORM
    1     2     2.373589E+03     1.133953E+00
    2     3     2.368923E+03     1.050547E+00
    3     4     2.366167E+03     7.486093E-01
    4     5     2.364338E+03     5.988088E-01
    5     6     2.362691E+03     5.236920E-01
    6     7     2.361469E+03     4.686596E-01
    7     8     2.360168E+03     4.120978E-01
    8     9     2.358962E+03     3.540827E-01
    9    10     2.358082E+03     2.897535E-01
   10    11     2.357598E+03     2.320233E-01
   11    12     2.357335E+03     1.953660E-01
   12    13     2.357025E+03     1.744273E-01
   13    14     2.356877E+03     1.522832E-01
   14    15     2.356603E+03     1.457189E-01
   15    16     2.356423E+03     1.453131E-01
   16    17     2.356053E+03     1.494561E-01
   17    18     2.355547E+03     1.455121E-01
   18    19     2.355401E+03     1.646963E-01
   19    20     2.355082E+03     1.914396E-01
   20    21     2.354492E+03     2.122671E-01
   21    22     2.354127E+03     1

Exception ignored in: <bound method IPythonKernel._clean_thread_parent_frames of <ipykernel.ipkernel.IPythonKernel object at 0x7f0dbf6e0650>>
Traceback (most recent call last):
  File "/home/jannik/storage/esmrmb-notebook/.venv/lib/python3.11/site-packages/ipykernel/ipkernel.py", line 781, in _clean_thread_parent_frames
    def _clean_thread_parent_frames(

KeyboardInterrupt: 


   69    70     2.330409E+03     1.870680E-01
   70    71     2.330014E+03     1.864428E-01
   71    72     2.329686E+03     1.848286E-01
